##  1) Exploration to find a manageable subnetwork

The full network (139k nodes, 5.3M edges) is too large
Community detection and SVD are hard to interpret at that scale

A subgraph allows:
- clear visualization
- meaningful interpretation
- advanced analysis

In this notebook we explore multiple candidate subnetworks and select a subset that is structurally meaningful and interpretable.

Our approach : Look at each brain region (group)
For each region:
- how many neurons?
- how many internal edges?
- how dense / connected is it?

We use this analysis to decide: **Which region is suitable for community detection ?**

In [3]:
import pandas as pd

# Load processed data from preprocessing.ipynb
edges = pd.read_csv("../data/preprocessed/processed_edges.csv")
nodes = pd.read_csv("../data/preprocessed/processed_nodes.csv")

print(nodes.shape)
print(edges.shape)


(139255, 4)
(3732460, 3)


#### What anatomical regions do we have, and how big are they?

In [15]:
group_sizes = (
    nodes["group"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "group", "group": "num_neurons"})
)

group_sizes.head(20)


,num_neurons,count
0,ME,43241
1,ME.LO,16312
2,LA,6446
3,ME.LOP,6071
4,LO.LOP,5863
5,NO_CONS,5074
6,LO,5052
7,GNG,4692
8,MB_CA.MB_ML,3139
9,AVLP,2825


In [16]:
candidate_groups = ["ME", "LO","LO.LOP","ME.LOP","SLP", "LOP", "LH","AVLP","LA","AOTU"]


For each group:
- keep only neurons in that group
- keep only edges where both pre and post are in the group

In [13]:
results = []

for g in candidate_groups:
    node_ids = nodes.loc[nodes["group"] == g, "id"]

    sub_edges = edges[
        edges["pre"].isin(node_ids) &
        edges["post"].isin(node_ids)
    ]

    results.append({
        "group": g,
        "num_nodes": node_ids.nunique(),
        "num_edges": len(sub_edges),
        "avg_weight": sub_edges["weight"].mean() if len(sub_edges) > 0 else 0
    })

pd.DataFrame(results)


,group,num_nodes,num_edges,avg_weight
0,ME,43241,705009,13.709324
1,LO,5052,60046,12.195250
2,LO.LOP,5863,2980,6.039597
3,ME.LOP,6071,3475,6.284604
4,SLP,1942,12851,10.203330
5,LOP,1921,20552,16.110987
6,LH,1132,10516,12.207969
7,AVLP,2825,114610,17.588744
8,LA,6446,7591,27.569095
9,AOTU,423,2908,15.908528


## Candidate Subnetwork Evaluation

### ME :Too Large to Analyze

- ~43,000 neurons with over 700,000 internal edges

- Far beyond what can be meaningfully visualized or interpreted.

- Spectral methods and community detection would be computationally heavy.

- Results would be difficult to interpret




### LA: Too Sparse or Weakly Connected


- ~6,400 neurons but only ~7,600 internal edges

- Very low internal connectivity relative to network size

- Community detection likely to be weak or noisy


### LO.LOP / ME.LOP

- ~6,000 neurons but fewer than ~3,500 internal edges

- Low edge density suggests limited intra-group communication

- Directed community structure is unlikely to be well defined

- Acceptable but Suboptimal

## LO

- ~5,000 neurons and ~60,000 internal edges

- Clearly structured and biologically meaningful

- Still relatively large for deep qualitative interpretation


### SLP / LH

- Between ~1,100 and ~1,900 neurons

- Moderate internal connectivity

- Smaller scope and less central to visual motion processing




### LOP: Best Candidate

- ~1,900 neurons with over 20,000 internal edges

- High edge density and strong directional connectivity

- Manageable size for visualization and spectral analysis

- Known to contain structured directional visual pathways

Decision: LOP Selected as the primary subnetwork

## 2)Extract the LOP subnetwork

Create a directed, weighted graph that contains:

- only LOP neurons

- only connections between LOP neurons

In [17]:
# Select only LOP neurons
lop_nodes = nodes[nodes["group"] == "LOP"].copy()

lop_nodes.shape


(1921, 4)

- Keep only edges where both endpoints are LOP neurons.
- Remove all external inputs/outputs

In [18]:
lop_ids = set(lop_nodes["id"])

lop_edges = edges[
    edges["pre"].isin(lop_ids) &
    edges["post"].isin(lop_ids)
].copy()

lop_edges.shape


(20552, 3)

In [19]:
print("LOP nodes:", lop_nodes["id"].nunique())
print("LOP edges:", len(lop_edges))
print("Average synapse weight:", lop_edges["weight"].mean())


LOP nodes: 1921
LOP edges: 20552
Average synapse weight: 16.11098676527832


In [20]:
#Build the graph object 
import networkx as nx

G_lop = nx.from_pandas_edgelist(
    lop_edges,
    source="pre",
    target="post",
    edge_attr="weight",
    create_using=nx.DiGraph()
)

G_lop.number_of_nodes(), G_lop.number_of_edges()


(1913, 20552)

In [21]:
# Save LOP subnetwork
lop_nodes.to_csv("../data/preprocessed/lop_nodes.csv", index=False)
lop_edges.to_csv("../data/preprocessed/lop_edges.csv", index=False)

print("LOP subnetwork saved.")


LOP subnetwork saved.
